In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mplconfig")

import numpy as np
import yt
import logging

yt.set_log_level("ERROR")
logging.getLogger("yt").setLevel(logging.ERROR)

plotfile = Path("diags/plt000250")
ds = yt.load(str(plotfile))

grid = ds.covering_grid(
    level=0,
    left_edge=ds.domain_left_edge,
    dims=ds.domain_dimensions,
)

Ex = grid[("boxlib", "Ex")].to_ndarray()
Ey = grid[("boxlib", "Ey")].to_ndarray()
Ez = grid[("boxlib", "Ez")].to_ndarray()

E = np.stack((Ex, Ey, Ez), axis=0)
E_magnitude = np.sqrt(Ex**2 + Ey**2 + Ez**2)

print(f"Loaded {plotfile}")
print(f"E shape: {E.shape}  # component, x, y, z")
print(f"E_magnitude shape: {E_magnitude.shape}")
print(f"E_magnitude min/max: {E_magnitude.min():.6g}, {E_magnitude.max():.6g}")

In [ ]:
import matplotlib.pyplot as plt

nx, ny, nz = E_magnitude.shape
ix = nx // 2
iz = nz // 2

y_edges = np.linspace(
    float(ds.domain_left_edge[1]),
    float(ds.domain_right_edge[1]),
    ny + 1,
)
y = 0.5 * (y_edges[:-1] + y_edges[1:])

plt.figure(figsize=(7, 4))
plt.plot(y * 1e6, Ex[ix, :, iz], label="Ex")
plt.plot(y * 1e6, Ey[ix, :, iz], label="Ey")
plt.plot(y * 1e6, Ez[ix, :, iz], label="Ez")
plt.plot(y * 1e6, E_magnitude[ix, :, iz], label="|E|", color="black", linewidth=2)
plt.xlabel("y (um)")
plt.ylabel("Electric field")
plt.title(f"Electric field along y at x-index {ix}, z-index {iz}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation, FFMpegWriter
from tqdm import tqdm

yt.set_log_level("ERROR")
logging.getLogger("yt").setLevel(logging.ERROR)

diag_dir = Path("diags")
plotfiles = sorted(
    diag_dir.glob("plt*"),
    key=lambda path: int(path.name.removeprefix("plt")),
)

stride = 1
frames = plotfiles[::stride]

def centerline_e_magnitude(path):
    frame_ds = yt.load(str(path))
    frame_grid = frame_ds.covering_grid(
        level=0,
        left_edge=frame_ds.domain_left_edge,
        dims=frame_ds.domain_dimensions,
    )
    ex = frame_grid[("boxlib", "Ex")].to_ndarray()
    ey = frame_grid[("boxlib", "Ey")].to_ndarray()
    ez = frame_grid[("boxlib", "Ez")].to_ndarray()
    emag = np.sqrt(ex**2 + ey**2 + ez**2)

    nx, ny, nz = emag.shape
    ix = nx // 2
    iz = nz // 2
    y_edges = np.linspace(
        float(frame_ds.domain_left_edge[1]),
        float(frame_ds.domain_right_edge[1]),
        ny + 1,
    )
    y = 0.5 * (y_edges[:-1] + y_edges[1:])
    return float(frame_ds.current_time), y, emag[ix, :, iz]

times = []
lines = []
for path in tqdm(frames):
    time, y, line_values = centerline_e_magnitude(path)
    times.append(time)
    lines.append(line_values)

times = np.asarray(times)
lines = np.asarray(lines)

fig, ax = plt.subplots(figsize=(7, 4))
line, = ax.plot(y * 1e6, lines[0], color="black", linewidth=2)
title = ax.set_title(f"|E| along y, t = {times[0]:.3e} s")
ax.set_xlabel("y (um)")
ax.set_ylabel("|E|")
ax.set_xlim(float(y[0] * 1e6), float(y[-1] * 1e6))
ax.set_ylim(0.0, float(lines.max() * 1.05))
ax.grid(True, alpha=0.3)

def update(frame_index):
    line.set_ydata(lines[frame_index])
    title.set_text(f"|E| along y, t = {times[frame_index]:.3e} s")
    return line, title

animation = FuncAnimation(fig, update, frames=len(lines), interval=50, blit=False)

video_path = Path("wave_freespace_run/e_magnitude_along_y.mp4")
animation.save(video_path, writer=FFMpegWriter(fps=20, bitrate=1800), dpi=150)
plt.close(fig)

print(f"Wrote {video_path}")
print(f"Frames: {len(lines)}")